In [1]:
import duckdb
import pandas as pd
from river import linear_model, preprocessing, metrics

In [2]:
query_yellow_2009 = """
WITH CTE_yellow_2009 AS (
    SELECT 
        CAST(Trip_Pickup_DateTime AS TIMESTAMP) AS pick_up_time,
        CAST(Trip_Dropoff_DateTime AS TIMESTAMP) AS drop_off_time,
        CAST(Passenger_Count AS INTEGER) AS passenger_count,
        CAST(Trip_Distance AS INTEGER) AS trip_distance,
        Payment_Type AS payment_type,
        CAST(Total_Amt AS FLOAT)   AS total_amount,
        Tip_Amt AS tip_amount,
        CAST(Tip_Amt AS FLOAT)   AS tip_amount,
        CAST( CASE Payment_Type
            WHEN  'Credit' THEN 1
            WHEN  'CREDIT' THEN 1
            ELSE 2
        END AS INTEGER) AS payment_category,
        CAST( CASE Tip_Amt
            WHEN  0.0 THEN 1
            ELSE 0
        END AS INTEGER) AS tip_category
    FROM 'C:/Users/ekadw/Documents/DATA/NY_Taxi/2009/yellow_taxi_2009/yellow_tripdata_*.parquet'
    WHERE Trip_Pickup_DateTime IS NOT NULL
        AND Trip_Dropoff_DateTime IS NOT NULL
        AND Passenger_Count >= 0
        AND Trip_Distance >= 0 
        AND Trip_Distance <= 50
        AND Payment_Type IS NOT NULL
        AND Total_Amt >= 0
        AND Tip_Amt >= 0
        AND Trip_Pickup_DateTime >= '2009-01-01' 
        AND Trip_Pickup_DateTime < '2010-01-01'
), CTE_duration_yellow_2009 AS (
    SELECT
        pick_up_time,
        drop_off_time,
        passenger_count,
        trip_distance,
        total_amount,
        payment_category,
        tip_category,
        DATE_DIFF('day', pick_up_time, drop_off_time) AS duration_days,
        EPOCH(drop_off_time - pick_up_time) AS duration_seconds
    FROM CTE_yellow_2009
    WHERE payment_category = 1
)

SELECT 
    passenger_count,
    trip_distance,
    total_amount,
    CAST(duration_seconds AS FLOAT)   AS duration_seconds,
    tip_category
FROM CTE_duration_yellow_2009
WHERE duration_days = 0
"""

#con = duckdb.connect()
#df_yellow_2009 = con.execute(query_yellow_2009).fetchdf()
#df_yellow_2009.head()

In [3]:
# Connect to DuckDB
con = duckdb.connect("yellow_2009.duckdb")

# Example parameters
batch_size = 500_000
offset = 0

while True:
    df = con.execute(f"""
        {query_yellow_2009}
        LIMIT {batch_size} OFFSET {offset}
    """).fetchdf()

    if df.empty:
        break

    # process df here
    offset += batch_size

In [6]:
from river import linear_model, preprocessing, metrics

model = preprocessing.StandardScaler() | linear_model.LogisticRegression()
metric = metrics.Accuracy()

for batch in con.execute(query_yellow_2009).fetch_record_batch():
    df = batch.to_pandas()

    X = df.drop(columns=["tip_category"])
    y = df["tip_category"]

    for xi, yi in zip(X.to_dict(orient="records"), y):
        y_pred = model.predict_one(xi)
        model.learn_one(xi, yi)
        metric.update(yi, y_pred)

    print("Processed batch, accuracy:", metric.get())

Processed batch, accuracy: 0.969352
Processed batch, accuracy: 0.969389
Processed batch, accuracy: 0.9695183333333334
Processed batch, accuracy: 0.9696045
Processed batch, accuracy: 0.969676
Processed batch, accuracy: 0.969718
Processed batch, accuracy: 0.9696707142857143
Processed batch, accuracy: 0.969594875
Processed batch, accuracy: 0.9695737777777778
Processed batch, accuracy: 0.9695547
Processed batch, accuracy: 0.9695495454545454
Processed batch, accuracy: 0.9695396666666667
Processed batch, accuracy: 0.9695428461538461
Processed batch, accuracy: 0.9695460714285714
Processed batch, accuracy: 0.9695593333333333
Processed batch, accuracy: 0.969560875
Processed batch, accuracy: 0.9695795882352941
Processed batch, accuracy: 0.9696272777777778
Processed batch, accuracy: 0.9696817368421052
Processed batch, accuracy: 0.9697211
Processed batch, accuracy: 0.9697284761904762
Processed batch, accuracy: 0.9697355
Processed batch, accuracy: 0.9697342173913044
Processed batch, accuracy: 0.969

Since dataset highly imbalance, the accuracy metric is less accurate because model tend to predict the majority. For that reason, we can change the metric by focusing on balancing the imbalance dataset.

In [ ]:
from river import linear_model, preprocessing, metrics

model = preprocessing.StandardScaler() | linear_model.LogisticRegression()
metric = metrics.BalancedAccuracy()   # better for imbalance

for batch in con.execute(query_yellow_2009).fetch_record_batch():
    df = batch.to_pandas()

    X = df.drop(columns=["tip_category"])
    y = df["tip_category"]

    for xi, yi in zip(X.to_dict(orient="records"), y):
        y_pred = model.predict_one(xi)
        model.learn_one(xi, yi)
        metric.update(yi, y_pred)

    print("Processed batch, balanced accuracy:", metric.get())


Processed batch, balanced accuracy: 0.5001508704244862
Processed batch, balanced accuracy: 0.5001902493027736
Processed batch, balanced accuracy: 0.5002146523773596
Processed batch, balanced accuracy: 0.5001856682632837
Processed batch, balanced accuracy: 0.5001685451341861
Processed batch, balanced accuracy: 0.5001634486078645
Processed batch, balanced accuracy: 0.5001777158014638
Processed batch, balanced accuracy: 0.5001902210855368
Processed batch, balanced accuracy: 0.5002270702429364
Processed batch, balanced accuracy: 0.5002261360223541
Processed batch, balanced accuracy: 0.5002170493212446
Processed batch, balanced accuracy: 0.5002151275709655
Processed batch, balanced accuracy: 0.5002024134647688
Processed batch, balanced accuracy: 0.500209309617351
Processed batch, balanced accuracy: 0.5002110089005969
Processed batch, balanced accuracy: 0.5002076659266773
Processed batch, balanced accuracy: 0.5002093394655317
Processed batch, balanced accuracy: 0.5002035632429928
Processed b

Try the manual weight

In [ ]:
from river import linear_model, preprocessing, metrics

# Example: if your dataset has many "low" and few "high",
# you can give more weight to "high".
weights = {"low": 1.0, "medium": 2.0, "high": 5.0}

model = preprocessing.StandardScaler() | linear_model.LogisticRegression(
    optimizer="adam",
    loss="log",
    l2=0.01,
    class_weights=weights
)

metric = metrics.BalancedAccuracy()

for batch in con.execute(query_yellow_2009).fetch_record_batch():
    df = batch.to_pandas()

    X = df.drop(columns=["tip_category"])
    y = df["tip_category"]

    for xi, yi in zip(X.to_dict(orient="records"), y):
        y_pred = model.predict_one(xi)
        model.learn_one(xi, yi)       # learns with class weights
        metric.update(yi, y_pred)

    print("Processed batch, balanced accuracy:", metric.get())


Try the tree based model that representative for imbalance dataset

In [ ]:
from river import tree, preprocessing, metrics

model = preprocessing.StandardScaler() | tree.HoeffdingTreeClassifier(
    grace_period=200,
    split_confidence=1e-5
)

metric = metrics.BalancedAccuracy()

for batch in con.execute(query_yellow_2009).fetch_record_batch():
    df = batch.to_pandas()

    X = df.drop(columns=["tip_category"])
    y = df["tip_category"]

    for xi, yi in zip(X.to_dict(orient="records"), y):
        y_pred = model.predict_one(xi)
        model.learn_one(xi, yi)
        metric.update(yi, y_pred)

    print("Processed batch, balanced accuracy:", metric.get())
